# <font color ='green'> Gemma cross-validation fine tuning with classification head
Original title: GEMMA_FineTuning_GemmaLoRa_Classification_Cyber_cases-new_dataset-Cross-validation.ipynb

### import libraries

In [ ]:
import json
import torch
import pandas as pd
from datasets import Dataset, load_dataset
from huggingface_hub import login
from peft import LoraConfig, PeftModel, prepare_model_for_kbit_training, get_peft_model
from peft import LoraConfig, get_peft_model
from transformers import (
    AutoModelForCausalLM,
    AutoModelForSequenceClassification,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainingArguments,
    pipeline,
    logging,
    Trainer,
    DataCollatorWithPadding,
    EarlyStoppingCallback)
import bitsandbytes as bnb
import os
import gc
from sklearn.metrics import f1_score, precision_score, recall_score, confusion_matrix, classification_report, roc_auc_score, confusion_matrix
from sklearn.model_selection import train_test_split, StratifiedKFold
import numpy as np
import matplotlib.pyplot as plt
import re
import evaluate
from datetime import datetime
import time

from dotenv import dotenv_values
config = dotenv_values(".env")  

from sklearn.utils.class_weight import compute_class_weight
from torch.nn import CrossEntropyLoss

In [ ]:
# Data cleaning function - combined
def standardization(sent: str) -> str:
    '''
    Input: raw reviews (string)
    Output: cleaned & standardized reviews (string)
    '''
    # Convert to lowercase, remove unwanted patterns, and remove non-alphanumeric characters
    sent = re.sub(r'[^0-9a-zA-Z-ZäöüÄÖÜßéóƒÚâèåèñéçýáúåí\s]', '', sent.lower())  
    # Remove specific MAUDE patterns
    sent = re.sub(r'\(b\)\(6\)|\(b\) \(6\)|\(b\)\(4\)|\(b\) \(4\)|\[rs\]\.\\n', '', sent)
    # Replace multiple spaces and strip leading/trailing whitespaces
    sent = re.sub(r'\s+', ' ', sent).strip()  
    
    return sent

def clean_dataframe(df: pd.DataFrame) -> pd.DataFrame:
    '''
    Input: DataFrame with columns 'Text' and 'Cyber_related'
    Output: Cleaned DataFrame with standardized text
    '''
    # Standardize text by applying the standardization function to each row
    df["text"] = df["text"].apply(standardization) 

    return df

In [ ]:
metric = evaluate.combine(["accuracy", "f1", "precision", "recall"])

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    probabilities = torch.nn.functional.softmax(torch.tensor(predictions), dim=1).numpy()
    
    predictions = np.argmax(predictions, axis=1)  # Convert probabilities to predicted labels [0 or 1]
    roc_auc = roc_auc_score(labels, probabilities[:, 1])  # Use probabilities of the positive clas
    print(roc_auc)

    dict_metric = metric.compute(predictions=predictions, references=labels)
    dict_metric.update({'roc_auc': roc_auc})
    return dict_metric

In [ ]:
########################################################################

## Model Parameters
learning_rate = 2e-5
num_train_epochs = 5
weight_decay = 0.01
early_stop_patience = 2 # increase a bit
early_stop_threshold = 0.01 # increase a bit
max_words            = 2000
#early_stopping = EarlyStoppingCallback(early_stopping_patience=2, early_stopping_threshold=0.0)

# Model name
model_name_str = "google/gemma-2-2b-it"


## Output folder
# Define model name and timestamp
unique_model_str = 'Gemma_Classification_weighted' # CHANGE PER SCRIPT
timestamp = datetime.now().strftime('%Y-%m-%d_%H-%M')  # Ensures uniqueness

# Define output folders
output_base_path = 'results/'
unique_name_date = f"{unique_model_str}_early_stop_{early_stop_patience}_{early_stop_threshold}_max_n_{max_words}_{timestamp}"
output_folder = os.path.join(output_base_path, unique_name_date)

os.makedirs(output_folder, exist_ok=True) # Create directories
print(f"Model results will be saved in: {output_folder}") # Print directories for verification


# TO DO LATER: delete checkpoints

## Load and preprocess dataset
data_folder = 'data/'
data_file = 'cybersecurity_annotated_data.pq'
df = pd.read_parquet(os.path.join(data_folder, data_file))
df = clean_dataframe(df[['ID', 'text', 'label']].dropna())
df['text'] = df['text'].apply(lambda x : ' '.join(x.split(' ')[:max_words]))
df.head()

In [ ]:
## Model config

model_name_str = "google/gemma-2-2b-it"

# Tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name_str) 
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

# Model configuration
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16)

def initialize_model():
    model = AutoModelForSequenceClassification.from_pretrained(
        model_name_str, 
        num_labels = 2, 
        id2label   = {0:'noCyber',1:'Cyber'},
        label2id   = {'noCyber':0,'Cyber':1},
        quantization_config=bnb_config, 
        device_map = {"": 0})
    
    model.gradient_checkpointing_enable()
    model = prepare_model_for_kbit_training(model)
    modules = find_all_linear_names(model)
    lora_config = LoraConfig(
        r=64,
        lora_alpha=32,
        target_modules=modules,
        lora_dropout=0.05,
        bias="none",
        task_type="SEQ_CLS")
    
    # Wrap model with LoRA and PEFT
    model = get_peft_model(model, lora_config)

    # Move model to CUDA explicitly
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)  # Ensure the model is on the correct device (GPU)

    return model

def find_all_linear_names(model):
    lora_modules = {name.split('.')[-1] for name, module in model.named_modules() if isinstance(module, bnb.nn.Linear4bit)}
    lora_modules.discard('lm_head')  # Remove if 16-bit compatibility is required
    return list(lora_modules)

def preprocess_function(examples):
    return tokenizer(examples["text"], truncation=True)

### Cross validation

In [ ]:
# Cross-Validation Script

# 5-Fold Stratified Cross-Validation
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Storage lists
metrics_per_fold = []
predictions_per_fold = []

for fold, (train_idx, test_idx) in enumerate(skf.split(df, df['label'])):
    start_time = time.time()  # Start timing
    print(f"Starting Fold {fold + 1}")
    
    # Prepare train/test splits
    train_df = df.iloc[train_idx].reset_index(drop=True)
    test_df = df.iloc[test_idx].reset_index(drop=True)

    train_dataset = Dataset.from_pandas(train_df)
    test_dataset = Dataset.from_pandas(test_df)

    tokenized_train = train_dataset.map(preprocess_function, batched=True)
    tokenized_test = test_dataset.map(preprocess_function, batched=True)
    
    # Initialize model
    model = initialize_model()
    model.print_trainable_parameters()
    
    # Define output directory for the fold
    output_dir = f"{output_folder}/model_output_fold_{fold + 1}"
    os.makedirs(output_dir, exist_ok=True)

    # ### Weighting - added from GEMMA_CV_FineTuning_Classification_weighted notebook
    
    ### Class weight
    class_weights = compute_class_weight('balanced', classes=np.unique(tokenized_train['label']), y=tokenized_train['label'])
    class_weights_tensor = torch.tensor(class_weights, dtype=torch.float)
    ### Modify loss function
    loss_function = CrossEntropyLoss(weight=class_weights_tensor)
    ### weighted training function
    class WeightedTrainer(Trainer):
        def compute_loss(self, model, inputs, num_items_in_batch=None, return_outputs=False):
            labels = inputs.get("labels")
            ### Get the device from the model
            device = model.device
            ### Ensure the class weights are on the same device as the model
            loss_function = CrossEntropyLoss(weight=class_weights_tensor.to(device))
            ### Get model outputs
            outputs = model(**inputs)
            logits = outputs.get("logits")
            ### Calculate the loss with class weights
            loss = loss_function(logits, labels)
            return (loss, outputs) if return_outputs else loss

    # Training arguments
    training_args = TrainingArguments(
        output_dir=output_dir,
        learning_rate=learning_rate, #defined at start
        per_device_train_batch_size=1,
        per_device_eval_batch_size=1,
        num_train_epochs=num_train_epochs, #defined at start
        weight_decay=weight_decay, #defined at start
        eval_strategy="epoch",
        save_strategy="epoch",
        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",
        logging_dir=f"{output_dir}/logs",
        logging_strategy="steps",  # Ensure logging occurs
        logging_steps=10,
        report_to='none')
    
    # Trainer
    trainer = WeightedTrainer(
        model=model,
        args=training_args,
        train_dataset=tokenized_train,
        eval_dataset=tokenized_test,
        tokenizer=tokenizer,
        data_collator=data_collator,
        compute_metrics=compute_metrics,
        callbacks=[
            EarlyStoppingCallback(early_stopping_patience=early_stop_patience, early_stopping_threshold=early_stop_threshold)])
    
    # Train the model
    trainer.train()
    trainer.save_model(output_dir)
    
    # Evaluate on test set
    metrics = trainer.evaluate(tokenized_test)
    print(f"Fold {fold + 1} Metrics: {metrics}")

    # Predict on the test set ## ADDED
    predictions = trainer.predict(tokenized_test)
    probabilities_logits = predictions.predictions  # Assuming these are raw logits
    probabilities = torch.nn.functional.softmax(torch.tensor(probabilities_logits), dim=-1).numpy()
    test_df["Probabilities"] = probabilities.tolist()  # Save all class probabilities
    test_df["Predicted_Label"] = predictions.predictions.argmax(axis=-1)  # Assuming classification
    test_df["Fold"] = fold  # Store fold information

    # Add fold info & time tracking
    fold_metrics = {
        'Fold': fold,
        'Time_Taken_Seconds': time.time() - start_time  # Track time per fold
    }
    # Combine evaluation metrics with fold info
    fold_metrics.update(metrics)
    print(fold_metrics)
    
    # Log metrics and predictions
    metrics_per_fold.append(fold_metrics)
    predictions_per_fold.append(test_df.copy())

    # avoiding GPU memory exhaustion - debugging
    torch.cuda.empty_cache()

# Convert to DataFrames
metrics_df = pd.DataFrame(metrics_per_fold)
predictions_df = pd.concat(predictions_per_fold, ignore_index=True)

# Save as Parquet
metrics_df.to_parquet(f"{output_folder}/cross_validation_metrics.pq", engine="pyarrow", index=False)
predictions_df.to_parquet(f"{output_folder}/cross_validation_predictions.pq", engine="pyarrow", index=False)

In [ ]:
# Print summary using a loop
print("\nCross-Validation Summary:")
for metric in ["eval_f1", "eval_precision", "eval_recall"]:
    print(f"Average {metric} Score: {metrics_df[metric].mean():.4f} ± {metrics_df[metric].std():.4f}")

print(f"Average Time per Fold: {metrics_df['Time_Taken_Seconds'].mean():.2f} sec ± {metrics_df['Time_Taken_Seconds'].std():.2f} sec")

In [ ]:
## Reading output back in at the end
# all predictions to a CSV file
all_predictions = pd.read_parquet(f"{output_folder}/cross_validation_predictions.pq")

# metrics for all folds
cv_results = pd.read_parquet(f"{output_folder}/cross_validation_metrics.pq") #cross_validation_results.csv

In [ ]:
## To be adapted given new tracking df?
# Metrics for plotting
metrics = ["eval_f1", "eval_precision", "eval_recall"]
metric_means = cv_results[metrics].mean()
metric_stds = cv_results[metrics].std()

# Bar chart with error bars
plt.figure(figsize=(10, 6))
x_labels = metrics
x_pos = np.arange(len(x_labels))

# Plot bars with error bars
means = metric_means.values
stds = metric_stds.values
colors = ['skyblue', 'orange', 'red']
plt.bar(x_pos, means, yerr=stds, capsize=5, color=colors, alpha=0.8, edgecolor='black')

# Formatting the plot
plt.xticks(x_pos, metrics, fontsize=12)
plt.ylabel('Scores', fontsize=14)
plt.title('Cross-Validation Metrics for GEMMA-2-2b finetuned Seq Classif', fontsize=16)
plt.ylim(0, 1.1)  # Extend the y-axis slightly for better spacing
plt.grid(axis='y', linestyle='--', alpha=0.6)
plt.tight_layout()

# Display values on bars
for i, (mean, std) in enumerate(zip(means, stds)):
    plt.text(i, mean + std + 0.03, f'{mean:.2f} ± {std:.2f}', 
             ha='center', fontsize=11, fontweight='bold')

# Show the plot
plt.show()